# Delimiter-guided non-binary top-down oracle with Stanza trees

This notebook mirrors the bottom-up notebook, but implements the top-down instantiation of the unified constituency transition system.

The pipeline:

1. read a PTB-style tree with `read_trees`,
2. extract the sentence,
3. derive the static top-down oracle,
4. apply the oracle to reconstruct a Stanza `Tree`,
5. check exact reconstruction against the original tree.

The top-down transition inventory used here is:

\[
\textsc{Shift},\ \textsc{Nt}(X),\ \textsc{u-Reduce},\ \textsc{Reduce},\ \textsc{Finish}.
\]

Unlike the bottom-up system, top-down parsing does not need explicit `D-SHIFT` or `D-REDUCE`.  A phrasal `NT(X)` opens a pending constituent before its children are built.  Internally, the applicator represents this by pushing a delimiter plus the label marker, so that phrase-level `REDUCE` can recover the full child sequence from the stack without an arity-specific action.

For preterminals, the same surface action `NT(POS)` is used, followed by `SHIFT(word)` and `U-REDUCE`.  The unary reduction constructs the preterminal node and removes the temporary marker machinery.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, List, Optional, Sequence, Tuple, Union

from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees

OracleOp = Tuple[str, Optional[str]]


/Users/jungyeul/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jungyeul/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tree utilities

The notebook uses Stanza `Tree` objects throughout.  The helper functions are intentionally the same as in the bottom-up notebook, so that the two notebooks can be compared directly.


In [2]:
def parse_ptb(ptb_string: str) -> Tree:
    """Parse one PTB-style bracketed tree into a Stanza constituency Tree."""
    trees = list(read_trees(ptb_string, use_tqdm=False))
    if not trees:
        raise ValueError(f"No tree found in string: {ptb_string!r}")
    if len(trees) != 1:
        raise ValueError(f"Expected exactly one tree, found {len(trees)}.")
    return trees[0]


def label_of(tree: Tree) -> str:
    """Return the label stored on a Stanza Tree node."""
    return tree.label


def children_of(tree: Tree) -> List[Tree]:
    """Return the children of a Stanza Tree node as a list."""
    return list(tree.children or [])


def make_tree(label: str, children: Optional[Sequence[Tree]] = None) -> Tree:
    """Construct a Stanza Tree node.

    Stanza versions expose the same logical constructor, but this helper keeps
    construction localized in case a local environment uses keyword-only fields.
    """
    children = list(children or [])
    try:
        return Tree(label, children)
    except TypeError:
        return Tree(label=label, children=children)


def is_leaf(tree: Tree) -> bool:
    return len(children_of(tree)) == 0


def is_preterminal(tree: Tree) -> bool:
    children = children_of(tree)
    return len(children) == 1 and is_leaf(children[0])


def terminal_word(preterminal: Tree) -> str:
    """Return the terminal word dominated by a preterminal."""
    if not is_preterminal(preterminal):
        raise ValueError(f"Expected a preterminal, found: {tree_to_ptb(preterminal)}")
    return label_of(children_of(preterminal)[0])


def get_sentence(tree: Tree) -> List[str]:
    """Return the terminal yield of a Stanza Tree."""
    if is_leaf(tree):
        return [label_of(tree)]

    words: List[str] = []
    for child in children_of(tree):
        words.extend(get_sentence(child))
    return words


def tree_to_ptb(tree: Tree) -> str:
    """Return a single-line PTB string from a Stanza Tree."""
    if is_leaf(tree):
        return label_of(tree)
    return f"({label_of(tree)} {' '.join(tree_to_ptb(child) for child in children_of(tree))})"


def same_tree(left: Tree, right: Tree) -> bool:
    """Exact structural comparison through the canonical PTB string form."""
    return tree_to_ptb(left) == tree_to_ptb(right)


## Static oracle

The top-down oracle follows the tree recursively.

For every internal node, the oracle first emits `NT(label)`, then visits the children from left to right, and finally emits `REDUCE`.

For every preterminal, the oracle emits:

```python
NT(POS)
SHIFT(word)
U-REDUCE
```

The final action is always `FINISH`.


In [3]:
def get_oracle_topdown(tree: Tree) -> List[OracleOp]:
    """Return the non-binary top-down oracle for a gold tree.

    The oracle is deterministic for a fixed tree.  It does not encode arity in
    phrase-level reduction actions.  The final action is always FINISH.
    """
    oracle: List[OracleOp] = []

    def visit(node: Tree) -> None:
        if is_leaf(node):
            raise ValueError(
                "Top-down oracle should visit terminals through their preterminal parent. "
                f"Unexpected leaf: {label_of(node)!r}"
            )

        if is_preterminal(node):
            oracle.append(("NT", label_of(node)))
            oracle.append(("SHIFT", terminal_word(node)))
            oracle.append(("U-REDUCE", None))
            return

        oracle.append(("NT", label_of(node)))
        for child in children_of(node):
            visit(child)
        oracle.append(("REDUCE", None))

    visit(tree)
    oracle.append(("FINISH", None))
    return oracle


## Oracle application

The applicator reconstructs a Stanza `Tree` from the sentence and oracle.  Internally, the stack contains completed `Tree` objects and label markers.  Dummy markers represent delimiter boundaries and are never included in the final tree.

For top-down parsing, `NT(X)` pushes a delimiter and a label marker:

```text
D | NT(X)
```

After the children have been constructed, `REDUCE` applies to:

```text
D | NT(X) | child1 | ... | childk
```

and creates:

```text
X(child1, ..., childk)
```

For preterminals, `U-REDUCE` applies to:

```text
D | NT(POS) | word
```

and creates:

```text
POS(word)
```


In [4]:
@dataclass(frozen=True)
class LabelMarker:
    label: str
    dummy: bool = False

    def __repr__(self) -> str:
        return "NT()" if self.dummy else f"NT({self.label})"


StackItem = Union[Tree, LabelMarker]


def _is_dummy(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and item.dummy


def _is_real_marker(item: StackItem) -> bool:
    return isinstance(item, LabelMarker) and not item.dummy


def _item_to_string(item: StackItem) -> str:
    if isinstance(item, LabelMarker):
        return repr(item)
    return tree_to_ptb(item)


def apply_oracle_topdown(
    sentence: Sequence[str],
    oracle: Sequence[OracleOp],
    *,
    verbose: bool = False,
) -> Tree:
    """Apply a delimiter-guided top-down oracle and return the reconstructed Stanza Tree."""

    stack: List[StackItem] = []
    word_index = 0
    finished = False
    final_tree: Optional[Tree] = None

    def stack_to_string() -> str:
        return "[ " + " | ".join(_item_to_string(item) for item in stack) + " ]" if stack else "[]"

    def show(step: int, action: str, value: Optional[str], when: str) -> None:
        if not verbose:
            return
        op = action if value is None else f"{action}({value})"
        remaining = " ".join(sentence[word_index:])
        print(f"{step:03d} {when:<6} {op:<18} stack={stack_to_string()} buffer=[ {remaining} ]")

    def shift(action_value: Optional[str]) -> None:
        nonlocal word_index

        if word_index >= len(sentence):
            raise ValueError("SHIFT exceeds sentence length.")

        word = sentence[word_index]
        if action_value is not None and action_value != word:
            raise ValueError(f"SHIFT mismatch: oracle has {action_value!r}, sentence has {word!r}.")

        stack.append(make_tree(word, []))
        word_index += 1

    def open_nt(label: str) -> None:
        # In the top-down instantiation, NT(X) opens both the label and its
        # left boundary.  The delimiter is internal to the applicator and does
        # not appear as a separate oracle action.
        stack.append(LabelMarker("", dummy=True))
        stack.append(LabelMarker(label))

    def unary_reduce() -> None:
        # Expected top-down preterminal pattern:
        #   ... | D | NT(P) | w
        if len(stack) < 3:
            raise ValueError(f"Invalid U-REDUCE configuration: {stack_to_string()}")

        if not isinstance(stack[-1], Tree) or not _is_real_marker(stack[-2]) or not _is_dummy(stack[-3]):
            raise ValueError(f"Invalid U-REDUCE configuration: {stack_to_string()}")

        child = stack.pop()
        marker = stack.pop()
        delimiter = stack.pop()

        assert isinstance(child, Tree)
        assert isinstance(marker, LabelMarker)
        assert isinstance(delimiter, LabelMarker)

        if not is_leaf(child):
            raise ValueError(f"U-REDUCE only applies to terminals, found: {tree_to_ptb(child)}")

        stack.append(make_tree(marker.label, [child]))

    def reduce_phrase() -> None:
        # Expected top-down phrase pattern:
        #   prefix | D | NT(X) | child1 | ... | childk
        marker_index: Optional[int] = None

        # The reducible marker is the rightmost real marker such that all items
        # above it are completed trees and the item immediately below it is D.
        for index in range(len(stack) - 1, -1, -1):
            if not _is_real_marker(stack[index]):
                continue

            if index == 0 or not _is_dummy(stack[index - 1]):
                continue

            suffix = stack[index + 1 :]
            if suffix and all(isinstance(item, Tree) for item in suffix):
                marker_index = index
                break

        if marker_index is None:
            raise ValueError(f"REDUCE cannot find a top-down reduction domain: {stack_to_string()}")

        marker = stack[marker_index]
        assert isinstance(marker, LabelMarker)

        delimiter_index = marker_index - 1
        prefix = stack[:delimiter_index]
        children = stack[marker_index + 1 :]

        if not children or not all(isinstance(child, Tree) for child in children):
            raise ValueError(f"Invalid children for REDUCE: {stack_to_string()}")

        new_tree = make_tree(marker.label, children)  # type: ignore[arg-type]
        stack[:] = list(prefix)
        stack.append(new_tree)

    def finish() -> Tree:
        if word_index != len(sentence):
            raise ValueError(f"FINISH before consuming the sentence: {word_index}/{len(sentence)}")

        if len(stack) != 1 or not isinstance(stack[0], Tree):
            raise ValueError(f"FINISH requires exactly one completed tree: {stack_to_string()}")

        return stack[0]

    for step, (action, value) in enumerate(oracle, start=1):
        if finished:
            raise ValueError(f"Action after FINISH at step {step}: {action}")

        show(step, action, value, "before")

        if action == "SHIFT":
            shift(value)
        elif action == "D-SHIFT":
            raise ValueError("D-SHIFT is not used in the top-down instantiation.")
        elif action == "NT":
            if value is None:
                raise ValueError("NT requires a label.")
            open_nt(value)
        elif action == "U-REDUCE":
            unary_reduce()
        elif action == "REDUCE":
            reduce_phrase()
        elif action == "D-REDUCE":
            raise ValueError("D-REDUCE is not used in the top-down instantiation.")
        elif action == "FINISH":
            final_tree = finish()
            finished = True
        else:
            raise ValueError(f"Unknown action: {action}")

        show(step, action, value, "after")

    if not finished or final_tree is None:
        raise ValueError("Oracle ended without FINISH.")

    return final_tree


## End-to-end helpers

`run_pipeline` is the main function used in the examples. It returns the sentence, oracle, reconstructed tree, and exact-match result.


In [5]:
def format_oracle(oracle: Sequence[OracleOp]) -> str:
    """Format an oracle as one action per line."""
    lines = []
    for i, (action, value) in enumerate(oracle, start=1):
        op = action if value is None else f"{action}({value})"
        lines.append(f"{i:03d}: {op}")
    return "\n".join(lines)


def action_counts(oracle: Sequence[OracleOp]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for action, _ in oracle:
        counts[action] = counts.get(action, 0) + 1
    return counts


def run_pipeline(ptb_string: str, *, verbose: bool = False) -> dict[str, Any]:
    gold_tree = parse_ptb(ptb_string)
    sentence = get_sentence(gold_tree)
    oracle = get_oracle_topdown(gold_tree)
    reconstructed_tree = apply_oracle_topdown(sentence, oracle, verbose=verbose)

    return {
        "sentence": sentence,
        "oracle": oracle,
        "gold_tree": gold_tree,
        "reconstructed_tree": reconstructed_tree,
        "exact_match": same_tree(gold_tree, reconstructed_tree),
        "action_counts": action_counts(oracle),
    }


## Minimal example

This example should match the top-down column in the paper:

```text
NT(ROOT)
NT(S)
NT(NP)
NT(NNP)
SHIFT(Ms.)
U-REDUCE
...
FINISH
```


In [6]:
ptb_string = "(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))"

result = run_pipeline(ptb_string)

print("Sentence:", result["sentence"])
print("Action counts:", result["action_counts"])
print("Exact reconstruction:", result["exact_match"])
print()
print(format_oracle(result["oracle"]))
print()
print("Reconstructed tree:")
print(tree_to_ptb(result["reconstructed_tree"]))

assert result["exact_match"]


Sentence: ['Ms.', 'Haag', 'plays', 'Elianti', '.']
Action counts: {'NT': 10, 'SHIFT': 5, 'U-REDUCE': 5, 'REDUCE': 5, 'FINISH': 1}
Exact reconstruction: True

001: NT(TOP)
002: NT(S)
003: NT(NP)
004: NT(NNP)
005: SHIFT(Ms.)
006: U-REDUCE
007: NT(NNP)
008: SHIFT(Haag)
009: U-REDUCE
010: REDUCE
011: NT(VP)
012: NT(VBZ)
013: SHIFT(plays)
014: U-REDUCE
015: NT(NP)
016: NT(NNP)
017: SHIFT(Elianti)
018: U-REDUCE
019: REDUCE
020: REDUCE
021: NT(.)
022: SHIFT(.)
023: U-REDUCE
024: REDUCE
025: REDUCE
026: FINISH

Reconstructed tree:
(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))


## Regression examples

These are the same example trees used in the bottom-up notebook. The assertion checks exact tree reconstruction, including punctuation.


In [7]:
examples = [
    "(TOP (S (NP (NNP Ms.) (NNP Haag)) (VP (VBZ plays) (NP (NNP Elianti))) (. .)))",
    "(TOP (S (NP (PRP We)) (VP (VBP 're) (VP (IN about) (S (VP (TO to) (VP (VB see) (SBAR (IN if) (S (NP (NN advertising)) (VP (VBZ works))))))))) (. .)))",
    "(TOP (S (NP (NP (JJ Influential) (NNS members)) (PP (IN of) (NP (DT the) (NNP House) (NNP Ways) (CC and) (NNP Means) (NNP Committee)))) (VP (VBD introduced) (NP (NP (NN legislation)) (SBAR (WHNP (WDT that)) (S (VP (MD would) (VP (VB restrict) (SBAR (WHADVP (WRB how)) (S (NP (DT the) (JJ new) (NN savings-and-loan) (NN bailout) (NN agency)) (VP (MD can) (VP (VB raise) (NP (NN capital)))))) (, ,) (S (VP (VBG creating) (NP (NP (DT another) (JJ potential) (NN obstacle)) (PP (TO to) (NP (NP (NP (DT the) (NN government) (POS 's)) (NN sale)) (PP (IN of) (NP (JJ sick) (NNS thrifts)))))))))))))) (. .)))",
]

for index, example in enumerate(examples, start=1):
    result = run_pipeline(example)
    print(
        f"example {index}: tokens={len(result['sentence'])}, "
        f"actions={len(result['oracle'])}, exact_match={result['exact_match']}, "
        f"counts={result['action_counts']}"
    )
    assert result["exact_match"]

example 1: tokens=5, actions=26, exact_match=True, counts={'NT': 10, 'SHIFT': 5, 'U-REDUCE': 5, 'REDUCE': 5, 'FINISH': 1}
example 2: tokens=9, actions=52, exact_match=True, counts={'NT': 21, 'SHIFT': 9, 'U-REDUCE': 9, 'REDUCE': 12, 'FINISH': 1}
example 3: tokens=37, actions=174, exact_match=True, counts={'NT': 68, 'SHIFT': 37, 'U-REDUCE': 37, 'REDUCE': 31, 'FINISH': 1}


## Optional verbose trace

Set `verbose=True` to inspect the stack and buffer before and after every transition.


In [9]:
_ = run_pipeline(ptb_string, verbose=True)


001 before NT(TOP)            stack=[] buffer=[ Ms. Haag plays Elianti . ]
001 after  NT(TOP)            stack=[ NT() | NT(TOP) ] buffer=[ Ms. Haag plays Elianti . ]
002 before NT(S)              stack=[ NT() | NT(TOP) ] buffer=[ Ms. Haag plays Elianti . ]
002 after  NT(S)              stack=[ NT() | NT(TOP) | NT() | NT(S) ] buffer=[ Ms. Haag plays Elianti . ]
003 before NT(NP)             stack=[ NT() | NT(TOP) | NT() | NT(S) ] buffer=[ Ms. Haag plays Elianti . ]
003 after  NT(NP)             stack=[ NT() | NT(TOP) | NT() | NT(S) | NT() | NT(NP) ] buffer=[ Ms. Haag plays Elianti . ]
004 before NT(NNP)            stack=[ NT() | NT(TOP) | NT() | NT(S) | NT() | NT(NP) ] buffer=[ Ms. Haag plays Elianti . ]
004 after  NT(NNP)            stack=[ NT() | NT(TOP) | NT() | NT(S) | NT() | NT(NP) | NT() | NT(NNP) ] buffer=[ Ms. Haag plays Elianti . ]
005 before SHIFT(Ms.)         stack=[ NT() | NT(TOP) | NT() | NT(S) | NT() | NT(NP) | NT() | NT(NNP) ] buffer=[ Ms. Haag plays Elianti . ]
005 after